# Gold — Daily Sales Metrics

Build `globalmart.gold.daily_sales_metrics` from **delivered** orders (on-time + late arrivals). Includes cumulative revenue, 7/30-day moving averages, day-over-day change, and within-month revenue rank.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.gold.daily_sales as daily_sales_module
importlib.reload(daily_sales_module)

from src.gold.daily_sales import (
    GoldDailySalesConfig,
    build_daily_sales_metrics,
    sample_month,
    run_daily_sales_metrics,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = GoldDailySalesConfig()
SAMPLE_YEAR = 2018
SAMPLE_MONTH = 1
print("Loaded from:", daily_sales_module.__file__)

In [ ]:
from pyspark.sql import functions as F

metrics = build_daily_sales_metrics(spark, config)
metrics.write.format("delta").mode("overwrite").saveAsTable(config.target_table)
print("Rows:", spark.table(config.target_table).count())
display(spark.table(config.target_table).orderBy(F.col("order_date").desc()).limit(5))

## Sample month — January 2018

All computed metrics for one calendar month.

In [ ]:
from pyspark.sql import functions as F

month_df = sample_month(spark.table(config.target_table), SAMPLE_YEAR, SAMPLE_MONTH)
display(month_df)

In [ ]:
import json

written = spark.table(config.target_table)
month_rows = [row.asDict() for row in sample_month(written, SAMPLE_YEAR, SAMPLE_MONTH).collect()]

report = {
    "task": "gold_daily_sales_metrics",
    "target_table": config.target_table,
    "daily_row_count": written.count(),
    "date_range": {
        "min": str(written.agg(F.min("order_date")).collect()[0][0]),
        "max": str(written.agg(F.max("order_date")).collect()[0][0]),
    },
    "window_sizes": {"short_ma_days": 7, "long_ma_days": 30},
    "sample_month": {"year": SAMPLE_YEAR, "month": SAMPLE_MONTH, "rows": month_rows},
    "top_revenue_day_in_sample_month": max(
        month_rows,
        key=lambda r: float(r["daily_revenue"] or 0),
        default=None,
    ),
}
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/gold_daily_sales.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)